# ftir_35 — Variation closure: what survived new HIPS holdouts and independent axes?

## tl;dr

The audit changes the calibration headline but strengthens the mechanism result. The exact
five-site launch contains **12,327 unique configuration×k rows per target**, not ~1,900 final
readouts. Only **one** row enters the 0.85–1.18 slope box at both Addis and Delhi under a
consistent estimator (analogs-440 AIRSpec-selected × deriv2, k=20), and it still reads
Deming **0.99x−2.22** at Addis and **1.13x−0.69** at Delhi (target R² 0.65/0.64); none reaches
target R² ≥0.70 at both. The screened Delhi winner's **0.87x−0.04** was Deming target fit,
while its quoted 0.86 was IMPROVE held-out TOR R²; 96.1% of Delhi was extrapolated. On the
26 newly reconstructed Delhi filters that locked winner falls to OLS **0.63x+1.11** (R²
0.62), and the common candidate loses Addis on 14 new filters (**0.55x−0.75**). HIPS
blank-line form and the raw-gain instrument epoch are closed: Addis York intercept is
−1.336 deployed, −1.328 linear, −1.326 quadratic, with no gain-residual association
(p=0.37). Two independent axes localize an additive HIPS-side component: Addis FTIR-EC vs
MA350 IR has intercept **+0.32**, but HIPS/MAC10 vs the same instrument is **+2.84**; Delhi
shows the same direction. AERONET Level-1.5 gives Addis a distinctive flat red AAE
(675–870 median **0.30** at AOD440≥0.4), but strict Level-2/U27 confirmation is unavailable.
Finally, loading/month-controlled potassium ion strongly tracks the Addis residual
(partial r **0.66**, permutation/FDR q<0.001), while the Al–Si–Ti–Fe dust index does not.
The most coherent reading is a HIPS-side additive bias plus real biomass/organic-linked
within-site structure, not one universal calibration correction.

## Context & Methods

This notebook consolidates the audit requested after the five-site sweep. It does not
select another winner. It reads the committed outputs of six reproducible analyses:

1. exact re-audit of all 61,635 five-site grid rows;
2. locked predictions on 54 reconstructed, never-screened HIPS targets;
3. calibration-set-specific HIPS blank-line, loading, and instrument-epoch diagnostics;
4. the pooled dry-season × HIPS interaction (ftir_24);
5. independent AERONET and MA350 optical checks; and
6. potassium/dust chemistry tests with loading and month controlled.

Reconstructed HIPS values remain provisional. They reproduce the shipped HIPS equation
and use the calibration line active on the analysis date, but they have not passed the
production pipeline's MDL/comment/uncertainty QC.

In [1]:
%matplotlib inline
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path('.')
VC = ROOT / 'output/tables/variation_closure'

grid_best = pd.read_csv(VC / 'five_site_grid_best_by_target.csv')
grid_joint = pd.read_csv(VC / 'five_site_grid_addis_etbi_joint.csv')
locked = pd.read_csv(VC / 'locked_reconstruction_summary.csv')
hips = pd.read_csv(VC / 'hips_epoch_loading_summary.csv')
loading = pd.read_csv(VC / 'hips_loading_correlations.csv')
season = pd.read_csv(ROOT / 'output/tables/ftir24/season_interactions.csv')
aeronet = pd.read_csv(ROOT / 'output/tables/aeronet/three_wavelength_summary.csv')
ma350 = pd.read_csv(VC / 'four_site_ma350_summary.csv')
chemistry = pd.read_csv(VC / 'residual_chemistry_tests.csv')

assert len(grid_joint[grid_joint['estimator'].eq('deming')]) == 1
assert len(grid_joint[grid_joint['estimator'].eq('ols')]) == 1
assert locked[locked['target'].str.endswith('_holdout')]['n'].sum() == 4 * (14 + 14 + 26)

## Data quality and grid audit

The saved JSONL contains 71,263 rows in total. The exact five-site launch contributes
61,635 rows: 12,327 unique configuration×k readouts at each of five targets. There are no
duplicate target keys in that launch subset. `heldout_R2` belongs to the IMPROVE TOR
calibration test; `target_R2` belongs to the SPARTAN city crossplot. The earlier Delhi
prose fused them, and the per-site table mixed Addis OLS with Delhi Deming.

In [2]:
audited = grid_best[
    grid_best['estimator'].eq('deming')
    & grid_best['minimum_target_R2'].eq(0.7)
][['target', 'n_eligible', 'cohort', 'cutoff', 'selection_space', 'spectra', 'k',
   'target_slope', 'target_intercept', 'target_R2', 'heldout_R2', 'extrap_pct']]
display(audited.round(3))

joint = grid_joint[grid_joint['estimator'].eq('deming')][[
    'cohort', 'cutoff', 'selection_space', 'spectra', 'k',
    'target_slope_addis', 'target_intercept_addis', 'target_R2_addis',
    'target_slope_delhi', 'target_intercept_delhi', 'target_R2_indh',
]]
display(joint.round(3))

,target,n_eligible,cohort,cutoff,selection_space,spectra,k,target_slope,target_intercept,target_R2,heldout_R2,extrap_pct
22,addis,39,ocec,440.0,raw,airspec,8.0,0.961,-1.552,0.720,0.924,NaN
26,chts,17,ocec,2000.0,raw,raw,13.0,1.110,0.005,0.706,0.855,14.6
30,etbi,14,ocec,1270.0,raw,raw,15.0,0.865,-0.511,0.820,0.966,3.8
34,indh,4,analogs,530.0,airspec,deriv2,20.0,0.873,-0.043,0.724,0.859,96.1
38,uspa,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,cohort,cutoff,selection_space,spectra,k,target_slope_addis,target_intercept_addis,target_R2_addis,target_slope_delhi,target_intercept_delhi,target_R2_indh
0,analogs,440.0,airspec,deriv2,20,0.988,-2.217,0.65,1.129,-0.692,0.639


## Locked HIPS confirmation

Configurations were frozen before the 54 values were reconstructed. This is a target-side
holdout from selection, not an independent analytical reference: the target still uses the
HIPS equation and lot blank line.

In [3]:
holdout = locked[
    locked['target'].str.endswith('_holdout')
    & locked['config'].isin(['addis_winner_k8', 'delhi_winner_k20',
                              'common_candidate_k20'])
][['config', 'target', 'n', 'ols_slope', 'ols_intercept', 'R2', 'RMSE',
   'mean_bias', 'extrap_pct', 'slope_ci_low', 'slope_ci_high']]
display(holdout.round(3))

,config,target,n,ols_slope,ols_intercept,R2,RMSE,mean_bias,extrap_pct,slope_ci_low,slope_ci_high
0,addis_winner_k8,addis_reconstructed_holdout,14,0.555,-0.161,0.733,2.143,-2.103,0.0,0.254,0.697
1,addis_winner_k8,etbi_reconstructed_holdout,14,0.659,0.282,0.638,0.459,-0.353,0.0,0.277,0.868
2,addis_winner_k8,indh_reconstructed_holdout,26,1.019,1.845,0.505,3.222,1.981,0.0,0.630,1.407
12,delhi_winner_k20,addis_reconstructed_holdout,14,0.453,-0.443,0.771,2.865,-2.830,100.0,0.187,0.558
13,delhi_winner_k20,etbi_reconstructed_holdout,14,0.289,0.153,0.307,1.242,-1.174,85.7,-0.023,0.473
14,delhi_winner_k20,indh_reconstructed_holdout,26,0.631,1.109,0.624,2.207,-1.576,88.5,0.470,0.874
18,common_candidate_k20,addis_reconstructed_holdout,14,0.550,-0.748,0.820,2.738,-2.711,100.0,0.316,0.668
19,common_candidate_k20,etbi_reconstructed_holdout,14,0.318,0.144,0.297,1.201,-1.129,85.7,-0.095,0.553
20,common_candidate_k20,indh_reconstructed_holdout,26,0.867,-0.454,0.645,2.184,-1.426,88.5,0.697,1.199


## HIPS-side variations

Blank-line form and the raw-gain instrument epoch do not explain the Addis intercept.
Calibration-set-specific refits are essential here: lot 251 itself used three deployed
lines over time, so a lot-pooled blank regression would mix instrument states.

In [4]:
hips_overall = hips[
    hips['grouping'].eq('overall')
    & hips['config'].isin(['addis_winner_k8', 'delhi_winner_k20'])
][['config', 'target', 'reference_variant', 'n', 'ols_slope', 'ols_intercept',
   'R2', 'york_slope', 'york_intercept', 'below_blank_r1_pct']]
display(hips_overall.round(3))

gain = loading[
    loading['predictor'].eq('instrument_gain')
    & loading['config'].isin(['addis_winner_k8', 'delhi_winner_k20'])
]
display(gain.round(4))

,config,target,reference_variant,n,ols_slope,ols_intercept,R2,york_slope,york_intercept,below_blank_r1_pct
0,addis_winner_k8,addis_augmented,deployed,253,0.844,-1.037,0.725,0.907,-1.336,43.083
1,addis_winner_k8,addis_augmented,linear,253,0.847,-1.038,0.724,0.909,-1.328,43.083
2,addis_winner_k8,addis_augmented,quadratic,253,0.844,-1.038,0.724,0.905,-1.326,43.083
60,addis_winner_k8,etbi_augmented,deployed,40,0.711,0.104,0.592,0.754,-0.001,7.500
61,addis_winner_k8,etbi_augmented,linear,40,0.707,0.150,0.593,0.748,0.052,7.500
62,addis_winner_k8,etbi_augmented,quadratic,40,0.706,0.150,0.593,0.748,0.053,7.500
81,addis_winner_k8,indh_augmented,deployed,178,1.661,-0.698,0.611,1.770,-1.378,15.169
82,addis_winner_k8,indh_augmented,linear,178,1.663,-0.623,0.613,1.768,-1.276,15.169
83,addis_winner_k8,indh_augmented,quadratic,178,1.651,-0.596,0.613,1.755,-1.247,15.169
396,delhi_winner_k20,addis_augmented,deployed,253,0.661,-0.976,0.595,0.724,-1.278,43.083


,config,target,predictor,n,spearman_r,p_value
2,addis_winner_k8,addis_augmented,instrument_gain,253,0.0561,0.3745
5,addis_winner_k8,etbi_augmented,instrument_gain,40,0.4417,0.0043
8,addis_winner_k8,indh_augmented,instrument_gain,178,-0.3642,0.0000
29,delhi_winner_k20,addis_augmented,instrument_gain,253,0.0886,0.1599
32,delhi_winner_k20,etbi_augmented,instrument_gain,40,0.3296,0.0378
35,delhi_winner_k20,indh_augmented,instrument_gain,178,0.1747,0.0197


## Season interaction

Separate season means reproduce ftir_15, but the pooled interaction adds a new result:
both raw and AIRSpec models have a significantly flatter dry-season slope. The corrected
model's mean bias is season-stable only because its intercept and slope move together; it
is not season-invariant in calibration geometry. Both February conventions agree.

In [5]:
season_primary = season[
    season['scope'].eq('all') & season['MAC'].eq(10)
][['model', 'convention', 'n', 'wet_slope', 'wet_intercept', 'dry_slope',
   'dry_intercept', 'dry_slope_delta', 'dry_slope_delta_p',
   'dry_intercept_delta', 'dry_intercept_delta_p', 'joint_p']]
display(season_primary.round(4))

,model,convention,n,wet_slope,wet_intercept,dry_slope,dry_intercept,dry_slope_delta,dry_slope_delta_p,dry_intercept_delta,dry_intercept_delta_p,joint_p
0,OCEC-800 raw k6,dry_feb,239,1.3523,-1.8065,0.8173,-0.4564,-0.5349,0.0013,1.3502,0.0827,0.0
4,OCEC-800 raw k6,belg_feb,239,1.3863,-2.0689,0.8464,-0.6462,-0.5399,0.0009,1.4228,0.0510,0.0
8,OCEC-800 AIRSpec k5,dry_feb,239,0.7458,-0.8231,0.4116,-0.0329,-0.3342,0.0000,0.7902,0.0246,0.0
12,OCEC-800 AIRSpec k5,belg_feb,239,0.7763,-1.0507,0.4337,-0.1523,-0.3426,0.0000,0.8985,0.0075,0.0


## Independent optical axes

AERONET is exploratory because the available inversion exports are Level 1.5 and contain
no U27 field. MA350 IR-880 is independent of both FTIR and HIPS calibration and more
directly localizes the additive mismatch.

In [6]:
aeronet_primary = aeronet[
    aeronet['subset'].eq('AOD440_ge_0.4')
][['city', 'n', 'AAE_440_675_median', 'AAE_675_870_median',
   'AAE_curvature_median', 'source_level_values']]
display(aeronet_primary.round(3))

ma350_exact = ma350[ma350['match'].eq('exact')][[
    'site', 'reference', 'n', 'ols_slope', 'ols_intercept',
    'intercept_ci_low', 'intercept_ci_high', 'R2',
    'deming_equal_error_slope', 'deming_equal_error_intercept',
]]
display(ma350_exact.round(3))

,city,n,AAE_440_675_median,AAE_675_870_median,AAE_curvature_median,source_level_values
1,Addis Ababa,100,1.716,0.297,1.350,lev15
4,Delhi (Gual Pahari),40,1.325,0.784,0.721,lev15
7,Delhi (Amity Gurgaon),1019,1.196,0.922,0.298,lev15
10,Delhi (New Delhi IMD),84,1.619,0.961,0.677,lev15
13,Beijing,718,1.617,0.956,0.622,lev15
16,Pasadena,23,1.187,0.736,0.186,lev15


,site,reference,n,ols_slope,ols_intercept,intercept_ci_low,intercept_ci_high,R2,deming_equal_error_slope,deming_equal_error_intercept
0,ETAD,FTIR_EC,173,0.618,0.320,0.010,0.629,0.868,0.645,0.113
1,ETAD,HIPS_BC_MAC10,173,0.266,2.835,2.652,3.017,0.778,0.272,2.795
4,CHTS,FTIR_EC,65,0.963,0.358,0.050,0.667,0.566,1.386,-0.158
5,CHTS,HIPS_BC_MAC10,65,0.813,0.450,0.296,0.604,0.788,0.905,0.337
8,INDH,FTIR_EC,24,0.653,0.942,-1.607,3.491,0.529,0.863,-0.887
9,INDH,HIPS_BC_MAC10,24,0.481,2.093,1.493,2.694,0.917,0.489,2.021
12,USPA,FTIR_EC,65,0.386,0.269,0.150,0.388,0.259,0.592,0.127
13,USPA,HIPS_BC_MAC10,65,0.630,0.113,0.049,0.177,0.761,0.689,0.072


## Chemistry attribution

The partial tests residualize both chemistry and calibration residual on HIPS loading and
cyclic month terms, then use a 5,000-permutation two-sided test with within-site BH-FDR.
Potassium/organic axes survive at Addis; the joint mineral-dust axis does not.

In [7]:
chem_focus = chemistry[
    ((chemistry['config'].eq('addis_winner_k8')
      & chemistry['target'].eq('addis_augmented'))
     | (chemistry['config'].eq('delhi_winner_k20')
        & chemistry['target'].eq('indh_augmented')))
][['config', 'target', 'predictor', 'n', 'partial_r', 'permutation_p', 'fdr_q']]
display(chem_focus.round(4))

,config,target,predictor,n,partial_r,permutation_p,fdr_q
0,addis_winner_k8,addis_augmented,K_ion,188,0.6620,0.0002,0.0006
1,addis_winner_k8,addis_augmented,K_total,188,0.5805,0.0002,0.0006
2,addis_winner_k8,addis_augmented,Al,188,-0.1143,0.1150,0.1293
3,addis_winner_k8,addis_augmented,Si,188,-0.1419,0.0510,0.0765
4,addis_winner_k8,addis_augmented,Ti,188,-0.1424,0.0498,0.0765
5,addis_winner_k8,addis_augmented,Fe,188,-0.1326,0.0652,0.0838
6,addis_winner_k8,addis_augmented,dust_index,188,-0.0716,0.3331,0.3331
7,addis_winner_k8,addis_augmented,Kion_to_Al,188,0.3359,0.0012,0.0027
8,addis_winner_k8,addis_augmented,OC_to_EC,175,0.5257,0.0002,0.0006
63,delhi_winner_k20,indh_augmented,K_ion,27,0.2980,0.1248,0.2361


## Takeaways

- **Retract the Delhi near-perfect calibration as a confirmed result.** It was a screened,
  96%-extrapolated target fit and fails the new selection-independent Delhi holdout.
- **Keep the no-universal-calibration conclusion.** Exactly one unique grid row enters the
  slope box at both cities, retains material offsets/moderate R², and fails the new Addis
  holdout. Calibration choice alone does not reconcile the cities.
- **Close blank-line curvature and a simple instrument epoch step.** Correct calibration-set
  refits move the Addis York intercept by only 0.01 µg/m³; E2/E3 and gain tests are stable.
- **Revise the season claim.** Mean corrected bias is similar by season, but pooled
  season×HIPS interactions show a significantly flatter Dry slope under both raw and
  AIRSpec models and both February conventions.
- **The independent evidence points to HIPS optics plus composition.** MA350 localizes a
  large additive difference to HIPS, while potassium/OC axes—but not the dust index—explain
  residual structure at Addis. AERONET's red-flat signal is supportive but Level-1.5 only.
- **Still external/new-data blockers:** official HIPS release/QC for the 54 reconstructions,
  Level-2 AERONET with U27, the remaining 202 SPARTAN filter volumes, newer IMPROVE lots,
  per-filter deposit images/reverse-orientation interpretation, and collocated quartz TOR.